In [5]:
import glob, os, re
import pandas as pd

FOLDERS = ["data/harmonized", "premerge_cleaned_dataset", "cleaned_v2"]

# ---------- 1. load every file, one per year ----------
frames = {}
for folder in FOLDERS:
    for f in sorted(glob.glob(f"{folder}/*.csv")):
        year = int(re.search(r"(?<!\d)(20\d{2})(?!\d)", os.path.basename(f)).group(1))
        if year in frames:
            raise ValueError(f"Two files found for {year}: check {f}")

        d = pd.read_csv(f, low_memory=False)

        # warn if an existing year label disagrees with the file name, then drop it
        for col in ["year", "survey_year"]:
            if col in d.columns:
                if set(d[col].dropna().unique()) != {year}:
                    print(f"WARNING {os.path.basename(f)}: '{col}' had values "
                          f"{sorted(d[col].dropna().unique())}, file name says {year}")
                d = d.drop(columns=col)
        d.insert(0, "survey_year", year)

        # add uid if the file doesn't already have one
        if "uid" not in d.columns:
            idc = "ResponseId" if "ResponseId" in d.columns else None
            d.insert(1, "uid", f"{year}_" + (d[idc].astype(str) if idc else d.index.astype(str)))

        frames[year] = d
        print(f"{year}: {d.shape[0]:>6,} rows, {d.shape[1]:>3} cols   <- {os.path.basename(f)}")

# ---------- 2. check year label and id ----------
failed = False
print()
for y, d in sorted(frames.items()):
    problems = []
    if d["uid"].isna().any():
        problems.append("blank uids")
    if d["uid"].duplicated().any():
        problems.append(f"{d['uid'].duplicated().sum()} duplicate uids")
    if not d["uid"].astype(str).str.startswith(f"{y}_").all():
        problems.append(f"some uids don't start with '{y}_'")
    print(y, "OK" if not problems else "PROBLEMS: " + "; ".join(problems))
    failed |= bool(problems)

if failed:
    raise SystemExit("Fix the problems above before merging.")

# ---------- 3. merge ----------
combined = pd.concat([frames[y] for y in sorted(frames)], ignore_index=True, sort=False)
print("\nCombined:", combined.shape)
print("duplicate uids:", combined["uid"].duplicated().sum())      # should be 0
print(combined["survey_year"].value_counts().sort_index())

# ---------- 4. save locally ----------
os.makedirs("combined", exist_ok=True)
combined.to_csv("combined/all_years.csv", index=False)

2017: 12,891 rows,  45 cols   <- 2017_droppedcols.csv
2018: 47,702 rows,  37 cols   <- 2018_droppedcols.csv
2019: 55,823 rows,  37 cols   <- 2019_droppedcols.csv
2020: 34,756 rows,  35 cols   <- 2020_droppedcols.csv
2021: 46,844 rows,  34 cols   <- 2021_droppedcols.csv
2011:  2,150 rows,  15 cols   <- final_canonical_2011.csv
2012:  3,740 rows,  15 cols   <- final_canonical_2012.csv
2013:  5,360 rows,  15 cols   <- final_canonical_2013.csv
2014:  4,623 rows,  15 cols   <- final_canonical_2014.csv
2015: 14,857 rows,  21 cols   <- final_canonical_2015.csv
2016: 41,742 rows,  20 cols   <- final_canonical_2016.csv
2022: 38,071 rows,  37 cols   <- survey_2022_v2.csv
2023: 48,019 rows,  53 cols   <- survey_2023_v2.csv
2024: 23,435 rows,  56 cols   <- survey_2024_v2.csv
2025: 23,947 rows,  79 cols   <- survey_2025_v2.csv
2026: 13,507 rows,  65 cols   <- survey_2026_v2.csv

2011 OK
2012 OK
2013 OK
2014 OK
2015 OK
2016 OK
2017 OK
2018 OK
2019 OK
2020 OK
2021 OK
2022 OK
2023 OK
2024 OK
2025 OK
2

In [2]:
df = pd.read_csv("combined/all_years.csv", low_memory=False)
df.head()

,year,uid,ResponseId,survey_year,Country,Age,YearsCode,Industry,OrgSize,DevType,...,AITool__AIWorkNotInWorkflow,AIAgents__AIMemWantToUseNext6Months,AIAgents__AIOrchHaveWorkedWith,AIAgents__AIOrchWantToUseNext6Months,AIAgents__AIObsHaveWorkedWith,AIAgents__AIObsWantToUseNext6Months,AIAgents__AINoCodeHaveWorkedWith,AIAgents__AINoCodeWantToUseNext6Months,AIAgents__AIAutomateHaveWorkedWith,AIAgents__AIAutomateWantToUseNext6Months
0,2011,2011_5,5,2011.0,Other Asia,35-39,11+,Software Products,Start Up (1-25),"Executive (VP of Eng, CTO, CIO, etc.)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011,2011_7,7,2011.0,United States of America,20-24,2-5,Manufacturing,Mature Small Business (25-100),"Executive (VP of Eng, CTO, CIO, etc.)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011,2011_8,8,2011.0,United States of America,20-24,6-10,Web Services,"Fortune 1000 (1,000+)",Web Application Developer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011,2011_9,9,2011.0,Germany,35-39,11+,Software Products,Mid Sized (100-999),Desktop Application Developer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,2011_11,11,2011.0,United Kingdom,35-39,11+,Education,Mid Sized (100-999),IT Staff / System Administrator,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.shape

(417467, 161)

In [4]:
df.head().T

,0,1,2,3,4
year,2011,2011,2011,2011,2011
uid,2011_5,2011_7,2011_8,2011_9,2011_11
ResponseId,5,7,8,9,11
survey_year,2011.0,2011.0,2011.0,2011.0,2011.0
Country,Other Asia,United States of America,United States of America,Germany,United Kingdom
...,...,...,...,...,...
AIAgents__AIObsWantToUseNext6Months,NaN,NaN,NaN,NaN,NaN
AIAgents__AINoCodeHaveWorkedWith,NaN,NaN,NaN,NaN,NaN
AIAgents__AINoCodeWantToUseNext6Months,NaN,NaN,NaN,NaN,NaN
AIAgents__AIAutomateHaveWorkedWith,NaN,NaN,NaN,NaN,NaN
